# Session 17 — Automated Model Retraining System using Data Drift Detection

**Goal:** close the loop that Session 5 (detect drift) and Session 15 (alert on
drift) left open — when drift crosses a threshold, **automatically retrain and
redeploy**, entirely locally, with MLflow tracking every generation of the model.

## The loop

```
new data arrives -> check drift vs. reference -> drift above threshold?
    -> yes: retrain on the new data, evaluate, promote if better -> update reference
    -> no:  do nothing, keep serving the current model
```

## Prerequisites

```bash
pip install mlflow evidently scikit-learn
```
Runs entirely locally.

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from evidently import Report
from evidently.presets import DataDriftPreset

mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session17-drift-triggered-retraining")

## Step 1 — Initial model, trained on the "reference" data window

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame
df = df.rename(columns={"target": "target"})

reference_df = df.sample(frac=0.4, random_state=0)
X_ref, y_ref = reference_df.drop(columns="target"), reference_df["target"]
X_ref_train, X_ref_test, y_ref_train, y_ref_test = train_test_split(
    X_ref, y_ref, test_size=0.2, random_state=0)

with mlflow.start_run(run_name="gen0_initial_model") as run:
    model = RandomForestClassifier(n_estimators=150, random_state=0).fit(X_ref_train, y_ref_train)
    auc = roc_auc_score(y_ref_test, model.predict_proba(X_ref_test)[:, 1])
    mlflow.log_metric("test_auc", auc)
    mlflow.sklearn.log_model(model, artifact_path="model")
    current_model_run_id = run.info.run_id
    current_model = model
    print(f"gen0 model trained. test AUC: {auc:.4f}")

## Step 2 — Simulate incoming production batches over time

Each "day", a new batch of data arrives. Most days look like the reference
distribution; a few days include a deliberate shift, simulating a sensor/process
change in how the underlying measurements are produced.

In [ ]:
remaining_df = df.drop(reference_df.index)
rng = np.random.default_rng(0)
DRIFT_STARTS_AT_BATCH = 3

batches = np.array_split(remaining_df.sample(frac=1.0, random_state=1), 6)
print(f"{len(batches)} simulated daily batches of ~{len(batches[0])} rows each")

## Step 3 — The monitor-and-retrain loop

For each incoming batch: check drift against the *current* reference, and if the
drift threshold is crossed, retrain on the new data and promote the new model only
if it's at least as good as the one it's replacing (never silently downgrade).

In [ ]:
DRIFT_THRESHOLD_PCT_COLUMNS = 0.3  # retrain if >=30% of columns show drift

history = []

for day, batch in enumerate(batches):
    if day >= DRIFT_STARTS_AT_BATCH:
        # Simulate a shift: a measurement device recalibration inflates a few features
        batch = batch.copy()
        batch["mean radius"] = batch["mean radius"] * 1.15
        batch["mean texture"] = batch["mean texture"] + rng.normal(3, 1, size=len(batch))

    drift_report = Report([DataDriftPreset()])
    snapshot = drift_report.run(current_data=batch.drop(columns="target"),
                                 reference_data=reference_df.drop(columns="target"))
    drift_count_metric = snapshot.dict()["metrics"][0]
    pct_drifted = drift_count_metric["value"]["share"]

    current_auc = roc_auc_score(batch["target"], current_model.predict_proba(batch.drop(columns="target"))[:, 1])

    action = "none"
    if pct_drifted >= DRIFT_THRESHOLD_PCT_COLUMNS:
        # Retrain on reference + this batch (simulating a growing labeled window)
        retrain_df = pd.concat([reference_df, batch])
        X_re, y_re = retrain_df.drop(columns="target"), retrain_df["target"]
        X_re_train, X_re_test, y_re_train, y_re_test = train_test_split(
            X_re, y_re, test_size=0.2, random_state=day)

        with mlflow.start_run(run_name=f"gen{day+1}_retrained_on_drift") as run:
            candidate_model = RandomForestClassifier(n_estimators=150, random_state=0).fit(X_re_train, y_re_train)
            candidate_auc = roc_auc_score(y_re_test, candidate_model.predict_proba(X_re_test)[:, 1])
            mlflow.log_metric("test_auc", candidate_auc)
            mlflow.log_metric("pct_columns_drifted", pct_drifted)
            mlflow.sklearn.log_model(candidate_model, artifact_path="model")

            if candidate_auc >= current_auc - 0.01:  # allow a tiny tolerance, don't require strict improvement
                current_model = candidate_model
                current_model_run_id = run.info.run_id
                reference_df = batch  # the new "normal" becomes today's batch
                action = f"retrained + promoted (new AUC {candidate_auc:.4f})"
            else:
                action = f"retrained but NOT promoted (candidate AUC {candidate_auc:.4f} < current {current_auc:.4f})"

    history.append({"day": day, "pct_columns_drifted": round(pct_drifted, 3),
                     "current_auc": round(current_auc, 4), "action": action})
    print(f"day {day}: drift={pct_drifted:.0%}  auc={current_auc:.4f}  -> {action}")

## Step 4 — Review the retraining history

The MLflow-tracked run history (Session 1's `search_runs`) now doubles as an audit
trail: exactly when did the system retrain, why, and was the new model actually
promoted.

In [ ]:
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))

## Step 5 — Why the "don't silently downgrade" guard matters

Retraining on drifted data isn't automatically better — if the drift is a data
*quality* problem (a broken sensor) rather than a genuine distribution shift,
blindly promoting a model trained on it could make things worse. The `candidate_auc
>= current_auc - 0.01` check is a minimal safeguard; a real system would also gate
on Deepchecks integrity checks (Session 11) before ever retraining.

## What to try next

* Replace the fixed `DRIFT_THRESHOLD_PCT_COLUMNS` with a statistically justified
  threshold (e.g. based on historical false-positive rate of the drift test itself).
* Add a Deepchecks integrity gate (Session 11) before the retrain step, so a broken
  data pipeline can never trigger an automatic retrain.
* Wire this loop into Session 14's Cloud Scheduler pattern (or a cron job / Airflow
  DAG) to run automatically instead of manually looping over `batches` in a notebook.